# AURA Virtual Assistant
## Asistente con Gemini en Google Colab

**Instrucciones:**
1. Abre este notebook en Colab
2. Runtime > Change runtime type > T4 GPU
3. Ve a Secrets (🔑) y agrega `GEMINI_API_KEY`
4. Ejecuta las celdas en orden

**Funciones:**
- Diagnóstico real del proyecto AURA (sin rutas rotas)
- Chat interactivo con Gemini conectado al contexto
- Verificación de backend, modelos y tokens
- Health check del entorno Colab

In [ ]:
# @title 1. Configurar Gemini y dependencias
# @markdown Instala SDKs y configura Gemini desde Colab Secrets

from google.colab import userdata
import os

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print('✅ Gemini API Key configurada desde Secrets')
else:
    print('❌ No GEMINI_API_KEY en Colab Secrets')
    print('   Agrega el secreto o ejecuta: os.environ["GEMINI_API_KEY"]="tu_key"')

!pip install -q google-generativeai requests gradio

import google.generativeai as genai
import requests
import gradio as gr

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print('✅ Gemini SDK listo')
else:
    print('⚠️ Gemini no configurado; el chat no funcionará')

In [ ]:
# @title 2. Diagnóstico del entorno AURA
# @markdown Verifica archivos, backend y configuración sin asumir rutas

import os
import glob
from datetime import datetime

def diagnose():
    report = []
    report.append('=' * 60)
    report.append('DIAGNÓSTICO AURA')
    report.append('=' * 60)
    
    cwd = os.getcwd()
    report.append(f'\n📁 Directorio actual: {cwd}')
    
    candidates = {
        'merged_training_data': ['aura_merged_training.jsonl', 'training-data.jsonl', 'data/*.jsonl'],
        'training_notebook': ['AURA_Colab_Training.ipynb', 'training.ipynb'],
        'assistant_notebook': ['AURA_ Colab_Assistant.ipynb', 'assistant.ipynb'],
        'backend_server': ['ame_backend/src/main.py', 'AME/servidor_ame.py', 'servidor_ame.py'],
        'ai_router': ['ame_backend/src/ai_engine.py', 'plugins/ai_router.py'],
        'dashboard': ['AME/dashboard.html', 'dashboard.html'],
        'models': ['models', 'models/*'],
        'data_dir': ['data', 'data/*'],
    }
    
    found = {}
    for name, paths in candidates.items():
        for p in paths:
            matched = glob.glob(p)
            if matched:
                path = matched[0]
                if os.path.isfile(path):
                    found[name] = f'{path} ({os.path.getsize(path)/1024:.1f} KB)'
                else:
                    found[name] = f'{path}/'
                break
        else:
            found[name] = 'NO ENCONTRADO'
    
    report.append('\n🔍 Archivos y componentes:')
    for k, v in found.items():
        icon = '✅' if 'NO ENCONTRADO' not in str(v) else '❌'
        report.append(f'  {icon} {k}: {v}')
    
    report.append('\n🌐 Backends:')
    backend_urls = [
        ('Backend Local Flask', 'http://localhost:5000/health'),
        ('Backend AME', 'http://localhost:5000/api/status'),
    ]
    for name, url in backend_urls:
        try:
            r = requests.get(url, timeout=3)
            if r.status_code == 200:
                report.append(f'  ✅ {name}: ONLINE ({r.status_code})')
                try:
                    data = r.json()
                    report.append(f'     {str(data)[:200]}')
                except:
                    pass
            else:
                report.append(f'  ⚠️ {name}: HTTP {r.status_code}')
        except Exception as e:
            report.append(f'  ❌ {name}: {str(e)[:100]}')
    
    hf = os.environ.get('HF_TOKEN', '')
    gemini = os.environ.get('GEMINI_API_KEY', '')
    report.append(f'\n🔑 Tokens:')
    report.append(f'  HF_TOKEN: {"✅ Configurado" if hf else "❌ Falta"}')
    report.append(f'  GEMINI_API_KEY: {"✅ Configurado" if gemini else "❌ Falta"}')
    
    model_dirs = [d.rstrip('/') for d in glob.glob('models/*/')]
    report.append(f'\n🧠 Modelos locales:')
    if model_dirs:
        for d in model_dirs:
            report.append(f'  ✅ {d}')
    else:
        report.append('  ❌ Ninguno detectado en models/')
    
    report.append('\n' + '=' * 60)
    return '\n'.join(report)

print(diagnose())

In [ ]:
# @title 3. Chat Interactivo con Gemini
# @markdown Asistente AURA con contexto del proyecto y acceso al backend

import ipywidgets as widgets
from IPython.display import display, clear_output

if not os.environ.get('GEMINI_API_KEY'):
    print('❌ Configura GEMINI_API_KEY primero (Celda 1)')
else:
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    model = genai.GenerativeModel('gemini-1.5-flash')
    
    SYSTEM_PROMPT = (
        'Eres AURA, asistente del proyecto AURA. '
        'Ayudas con: estado del proyecto, entrenamiento de modelos, '
        'integración Colab/HF Spaces/backend, diagnóstico técnico y próximos pasos. '
        'Responde en español, conciso y práctico.'
    )
    
    def get_context():
        ctx = []
        for pattern in ['aura_merged_training.jsonl', 'training-data.jsonl', 'data/*.jsonl']:
            files = glob.glob(pattern)
            if files:
                f = files[0]
                try:
                    with open(f, 'r', encoding='utf-8') as fh:
                        lines = sum(1 for _ in fh)
                    ctx.append(f'Datos entrenamiento: {lines:,} muestras en {f}')
                except:
                    pass
                break
        
        for url in ['http://localhost:5000/health', 'http://localhost:5000/api/status']:
            try:
                r = requests.get(url, timeout=2)
                if r.status_code == 200:
                    ctx.append(f'Backend: ONLINE en {url}')
                    break
            except:
                continue
        
        model_dirs = [d.rstrip('/') for d in glob.glob('models/*/')]
        if model_dirs:
            ctx.append(f'Modelos: {", ".join([os.path.basename(d) for d in model_dirs])}')
        
        return '\n'.join(ctx) if ctx else 'Sin datos adicionales del entorno'
    
    output = widgets.Output()
    input_box = widgets.Text(
        placeholder='Escribe tu mensaje (status, help, salir)...',
        description='Tu:',
        layout={'width': '90%'}
    )
    send_btn = widgets.Button(description='Enviar', button_style='primary')
    clear_btn = widgets.Button(description='Limpiar', button_style='warning')
    
    def on_send(b):
        msg = input_box.value.strip()
        if not msg:
            return
        if msg.lower() in ('salir', 'exit', 'quit'):
            with output:
                print('\n👋 Hasta luego!')
            input_box.disabled = True
            send_btn.disabled = True
            return
        if msg.lower() == 'status':
            with output:
                print(get_context())
            input_box.value = ''
            return
        if msg.lower() == 'help':
            with output:
                print('Comandos: status, help, salir')
                print('O escribe cualquier pregunta para Gemini')
            input_box.value = ''
            return
        
        with output:
            print(f'👤 Tú: {msg}')
            print('🤔 AURA: ', end='', flush=True)
            try:
                ctx = get_context()
                full_prompt = f'Contexto AURA:\n{ctx}\n\nUsuario: {msg}\n\nResponde como AURA, en español, conciso.'
                response = model.generate_content(full_prompt)
                print(response.text)
            except Exception as e:
                print(f'❌ Error: {e}')
            print()
        input_box.value = ''
    
    send_btn.on_click(on_send)
    input_box.on_submit(on_send)
    
    def on_clear(b):
        output.clear_output()
    
    clear_btn.on_click(on_clear)
    
    display(widgets.HBox([input_box, send_btn, clear_btn]))
    display(output)
    
    print('✅ Chat listo. Escribe un mensaje y presiona Enter o Enviar.')
    print('   Comandos: status, help, salir')

In [ ]:
# @title 4. Configuración de Entrenamiento
# @markdown Parámetros reales detectados del proyecto

import os
import glob

training_files = []
for pattern in ['aura_merged_training.jsonl', 'training-data.jsonl', 'data/*.jsonl']:
    training_files.extend(glob.glob(pattern))

model_dirs = [d.rstrip('/') for d in glob.glob('models/*/')]

config = {
    'model_base': 'Qwen/Qwen2.5-1.5B-Instruct',
    'base_model_local': 'models/qwen-1.5b/' if os.path.exists('models/qwen-1.5b') else 'No descargado',
    'training_data': training_files[0] if training_files else 'No encontrado',
    'total_samples': 'Detectar del archivo',
    'quantization': '4-bit (Unsloth)',
    'lora_rank': 16,
    'batch_size': 4,
    'gradient_accumulation': 4,
    'effective_batch': 16,
    'epochs': 1,
    'learning_rate': '2e-4',
    'max_seq_length': 2048,
    'gpu': 'T4 (15 GB VRAM)',
    'estimated_time': '~15 min',
    'output_dir': 'aura_finetuned_merged',
    'gguf_quantization': 'q4_k_m',
}

print('📋 Configuración de Entrenamiento')
print('=' * 45)
for k, v in config.items():
    print(f'  {k}: {v}')

print('\n📝 Pasos:')
print('  1. Abre AURA_Colab_Training.ipynb')
print('  2. Runtime > T4 GPU')
print('  3. Run all (Ctrl+F9)')
print('  4. Modelo guardado en aura_finetuned_merged/')
print('  5. Subir a HF Hub con upload_training_to_hf.py')

In [ ]:
# @title 5. Health Check del Sistema
# @markdown Verifica Python, PyTorch, dependencias y servicios

import subprocess
import sys

def run_check(name, command):
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=15
        )
        output = result.stdout.strip()[:200]
        if not output and result.stderr:
            output = result.stderr.strip()[:200]
        return name, result.returncode == 0, output
    except Exception as e:
        return name, False, str(e)[:200]

checks = [
    ('Python', f'{sys.executable} --version'),
    ('PyTorch', f'{sys.executable} -c "import torch; print(torch.__version__)"'),
    ('Gradio', f'{sys.executable} -c "import gradio; print(gradio.__version__)"'),
    ('Google GenAI', f'{sys.executable} -c "import google.generativeai; print(\"OK\")"'),
    ('Requests', f'{sys.executable} -c "import requests; print(requests.__version__)"'),
]

print('🏥 Health Check del Sistema')
print('=' * 40)
for name, cmd in checks:
    _, ok, out = run_check(name, cmd)
    icon = '✅' if ok else '❌'
    print(f'  {icon} {name}: {out if ok else "FALLO"}')

print('\n🌐 Backends:')
for url in ['http://localhost:5000/health', 'http://localhost:5000/api/status']:
    try:
        r = requests.get(url, timeout=2)
        print(f'  ✅ {url}: HTTP {r.status_code}')
    except Exception as e:
        print(f'  ❌ {url}: {str(e)[:80]}')

print('\n💡 Si algo falla, instala con: pip install <paquete>')
print('   Para Gemini, configura GEMINI_API_KEY en Secrets (Celda 1)')